# Module 4: Query Intelligence + Evaluation

## What’s new?

We improve retrieval by:
- rewriting queries
- generating multiple query variations
- analyzing retrieval quality

## Why?

Retrieval quality depends heavily on the input query.


## Architecture


      User Query
         ↓
      Query Rewriting (LLM)
         ↓
      Multi-Query Expansion
         ↓
      Hybrid Retrieval
         ↓
      Reranking
         ↓
      LLM Answer

In [1]:
%pip install -qU langchain langchain-openai langchain-chroma langchain-text-splitters

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate

c:\Users\vammun01\OneDrive - Robert Half\repos\prod-rag\.venv\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


## Setup Azure OpenAI environment variables & LLM and embedding models 
Copy content from previous module

In [3]:
# Initialize environment variables for Azure OpenAI
api_key = (os.getenv("AZURE_OPENAI_API_KEY") or "").strip()
embedding_deployment = (os.getenv("AZURE_OPENAI_EMBEDDING_DEPLOYMENT_NAME") or "").strip()
endpoint = (os.getenv("AZURE_OPENAI_ENDPOINT") or "").strip()
llm_deployment = (
    os.getenv("AZURE_OPENAI_LLM_DEPLOYMENT_NAME")
    or os.getenv("AZURE_OPENAI_CHAT_DEPLOYMENT_NAME")
    or os.getenv("AZURE_OPENAI_DEPLOYMENT_NAME")
    or ""
).strip()


# Create embeddings client based on endpoint type
# Foundry project endpoint -> OpenAI-compatible v1 endpoint.
if "services.ai.azure.com" in endpoint and "/api/projects/" in endpoint:
    resource_root = endpoint.split("/api/projects/")[0]
    openai_endpoint = f"{resource_root}/openai/v1"
    embed_model = OpenAIEmbeddings(
        model=embedding_deployment,
        api_key=api_key,
        base_url=openai_endpoint,
    )
    embed_model.embed_query("endpoint-check")
    print("Embeddings client initialized OpenAI-compatible endpoint.")

    llm = ChatOpenAI(
        model=llm_deployment,
        api_key=api_key,
        base_url=openai_endpoint,
        temperature=0,
        max_tokens=None,
        timeout=None,
        max_retries=2,
    )
    print("LLM initialized via OpenAI-compatible endpoint.")
else:
    api_version = (os.getenv("AZURE_OPENAI_API_VERSION") or "2024-02-01").strip()
    embed_model = AzureOpenAIEmbeddings(
        model=embedding_deployment,
        azure_endpoint=endpoint,
        api_key=api_key,
        openai_api_version=api_version,
    )
    embed_model.embed_query("endpoint-check")
    print(f"Embeddings client initialized via Azure OpenAI endpoint (api_version={api_version}).")

    llm = AzureChatOpenAI(
        azure_deployment=llm_deployment,
        azure_endpoint=endpoint,
        api_key=api_key,
        api_version=api_version,
        temperature=0,
        max_tokens=None,
        timeout=None,
        max_retries=2,
    )
    print(f"LLM initialized via Azure OpenAI endpoint (api_version={api_version}).")

Embeddings client initialized OpenAI-compatible endpoint.
LLM initialized via OpenAI-compatible endpoint.


In [4]:
from langchain_core.documents import Document
documents = [
    Document(
        page_content="LangChain helps build RAG systems using LLMs and retrievers.",
        metadata={"source": "langchain.pdf", "page": 1}
    ),
    Document(
        page_content="LangGraph enables multi-step workflows and agent orchestration.",
        metadata={"source": "langgraph.pdf", "page": 2}
    ),
    Document(
        page_content="LangSmith provides observability and evaluation for LLM systems.",
        metadata={"source": "langsmith.pdf", "page": 3}
    ),
]

In [5]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=50
)

chunks = text_splitter.split_documents(documents)

print(f"Total chunks created: {len(chunks)}")

Total chunks created: 3


In [6]:
from langchain_community.retrievers import BM25Retriever
bm25_retriever = BM25Retriever.from_documents(chunks)
bm25_retriever.k = 3

In [7]:
from langchain_chroma import Chroma
# Embeddings and Persistent DB
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embed_model,
    persist_directory="./chroma_db"
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

In [8]:
def format_docs(docs):
    return "\n\n".join(
        f"Source: {doc.metadata['source']} (Page {doc.metadata.get('page', 'N/A')})\n{doc.page_content}"
        for doc in docs
    )

In [9]:
def hybrid_retrieve(query):
    bm25_docs = bm25_retriever.invoke(query)
    vector_docs = retriever.invoke(query)
    
    # Combine and deduplicate
    combined = bm25_docs + vector_docs
    
    unique_docs = {doc.page_content: doc for doc in combined}
    
    return list(unique_docs.values())

In [10]:
def simple_reranker(query, docs):
    def score(doc):
        return sum(word in doc.page_content.lower() for word in query.lower().split())
    
    ranked = sorted(docs, key=score, reverse=True)
    return ranked[:3]

## Problem Example

User Query:
"What does it do?"

This query is too vague for retrieval.

We need to transform it into something like:
"What does LangGraph do in LLM systems?"

In [11]:
def rewrite_query(query):
    rewrite_prompt = ChatPromptTemplate.from_template("""
    Rewrite the following query to be more specific and retrieval-friendly.

    Original Query:
    {query}

    Improved Query:
    """)
    
    response = llm.invoke(
        rewrite_prompt.invoke({"query": query})
    )
    
    return response.content.strip()

In [12]:
query = "What does it do?"
improved_query = rewrite_query(query)

print("Original:", query)
print("Rewritten:", improved_query)

Original: What does it do?
Rewritten: Improved Query:
What is the purpose and main function of **[the specific feature/tool/system]**, and what are the key actions it performs when used in **[this context or workflow]**?


## Multi-Query Generation

Instead of one query, generate multiple variations.

In [13]:
def generate_multi_queries(query):
    multi_prompt = ChatPromptTemplate.from_template("""
    Generate 3 different variations of the query for better document retrieval.

    Query:
    {query}
    
    Variations:
    """)
    
    response = llm.invoke(
        multi_prompt.invoke({"query": query})
    )
    
    queries = response.content.split("\n")
    return [q.strip("- ").strip() for q in queries if q.strip()]

## Combine Multi-Query Retrieval

In [14]:
def multi_query_retrieve(query):
    queries = generate_multi_queries(query)
    
    all_docs = []
    for q in queries:
        docs = hybrid_retrieve(q)
        all_docs.extend(docs)
    
    # Deduplicate
    unique_docs = {doc.page_content: doc for doc in all_docs}
    
    return list(unique_docs.values())

## Rerank the Combined Results

In [15]:
def full_retrieval_pipeline(query):
    rewritten = rewrite_query(query)
    docs = multi_query_retrieve(rewritten)
    reranked = simple_reranker(rewritten, docs)
    
    return reranked

In [18]:
prompt = ChatPromptTemplate.from_template("""
Answer the question using only the context below.

Include source citations in your answer.

Context:
{context}

Question:
{question}
""")

In [19]:
def ask_advanced_rag(query):
    docs = full_retrieval_pipeline(query)
    
    context = format_docs(docs)
    
    final_prompt = prompt.invoke({
        "context": context,
        "question": query
    })
    
    response = llm.invoke(final_prompt)
    
    return {
        "answer": response.content,
        "docs": docs
    }

In [21]:
result = ask_advanced_rag("What does it do?")

print("Answer:\n", result["answer"])

print("\nDocuments Used:")
for d in result["docs"]:
    print(d.metadata)

Answer:
 It provides **observability and evaluation for LLM systems** (LangSmith) and enables **multi-step workflows and agent orchestration** (LangGraph), while helping build **RAG systems using LLMs and retrievers** (LangChain) [langsmith.pdf, p.3; langgraph.pdf, p.2; langchain.pdf, p.1].

Documents Used:
{'source': 'langsmith.pdf', 'page': 3}
{'page': 2, 'source': 'langgraph.pdf'}
{'source': 'langchain.pdf', 'page': 1}
